# LiDAR Lenses Wave v0.6.10 Test Harness

Upload two files:
1. `lidar_lenses_wave_v0610.py`
2. `lidar_wave_test_plan_v0610.xlsx`

This harness compares the new geometry-fused edge score against raw anti-wave behavior.


In [ ]:
from google.colab import files
uploaded = files.upload()
print('uploaded:', list(uploaded.keys()))


In [ ]:
!python -V
!pip -q install pandas openpyxl pillow matplotlib


In [ ]:
import glob, os, json, zipfile, importlib.util, sys
from pathlib import Path
import pandas as pd
from PIL import Image, ImageDraw

engine_files = sorted(glob.glob('lidar_lenses_wave_v0610.py') or glob.glob('lidar_lenses_wave_v*.py'))
plan_files = sorted(glob.glob('lidar_wave_test_plan_v0610.xlsx') or glob.glob('*test_plan*.xlsx'))
assert engine_files, 'Upload lidar_lenses_wave_v0610.py'
assert plan_files, 'Upload lidar_wave_test_plan_v0610.xlsx'
ENGINE = engine_files[-1]
PLAN = plan_files[-1]
OUT = Path('v0610_test_outputs')
OUT.mkdir(exist_ok=True)
print('ENGINE:', ENGINE)
print('PLAN:', PLAN)

spec = importlib.util.spec_from_file_location('llw', ENGINE)
llw = importlib.util.module_from_spec(spec)
sys.modules['llw'] = llw
spec.loader.exec_module(llw)
print('engine imported')


In [ ]:
import numpy as np

def _rot(rx=0, ry=0, rz=0):
    return llw.make_rotation_matrix(rx, ry, rz)

def _prim(shape, center, half, color, piece_id, piece_type, rot=None, transparency=0.0):
    if rot is None:
        rot = _rot()
    return llw.Primitive(shape=shape, center=np.array(center, float), half_extents=np.array(half, float),
                         rotation_matrix=rot, inv_rotation_matrix=rot.T,
                         color=tuple(float(c) for c in color), piece_id=int(piece_id),
                         piece_type=str(piece_type), transparency=float(transparency))

def build_occluder_gate():
    prims=[]; pid=1
    prims.append(_prim('box',[0,-0.5,0],[7,0.5,7],[0.55,0.50,0.38],pid,'ground')); pid+=1
    # wall/gate posts
    for x in [-2.2,-1.4,-0.6,0.6,1.4,2.2]:
        prims.append(_prim('cylinder',[x,0.7,1.2],[0.06,0.7,0.06],[0.82,0.76,0.60],pid,'wood')); pid+=1
    prims.append(_prim('box',[0,1.1,1.2],[2.7,0.08,0.06],[0.82,0.76,0.60],pid,'wood')); pid+=1
    # transparent foliage / porous sheet
    prims.append(_prim('sphere',[0,1.35,1.25],[1.2,0.75,0.4],[0.25,0.70,0.25],pid,'foliage',transparency=0.55)); pid+=1
    prims.append(_prim('box',[0,0.7,2.8],[2.2,0.7,0.08],[0.35,0.35,0.35],pid,'stone')); pid+=1
    return llw.apply_material_prior_transparency(prims)

def scene_from_name(name):
    n = str(name).strip().lower()
    if n in ('demo','cabin','cabin_demo','mini_saloon'):
        return llw.apply_material_prior_transparency(llw._build_demo_scene())
    if n in ('material_targets','material_board','material'):
        return llw.apply_material_prior_transparency(llw._build_material_target_board_scene())
    if n in ('occluder_gate','occlusion'):
        return build_occluder_gate()
    return llw.apply_material_prior_transparency(llw._build_demo_scene())


In [ ]:
plan = pd.read_excel(PLAN, sheet_name='sweep')
plan = plan[plan.get('enabled', True).astype(bool)].copy()
print(plan[['run_id','scene','preset','edge_score_mode','edge_fusion_mode']])


In [ ]:
all_diag=[]; structure=[]; material_core=[]; material_filled=[]; boundary=[]
contact_paths=[]
for _, row in plan.iterrows():
    run_id = int(row.get('run_id', len(all_diag)+1))
    scene = str(row.get('scene','demo'))
    preset = str(row.get('preset','compact_diagnostic'))
    prims = scene_from_name(scene)
    run_dir = OUT / f'run_{run_id:02d}_{scene}_{preset}'
    run_dir.mkdir(exist_ok=True)
    overrides = {}
    for k in ['width','height','rays_per_pixel','stack','pilot_rays','edge_anti_min','adaptive_edge_percentile','edge_anti_max']:
        if k in row and pd.notna(row[k]):
            overrides[k] = int(row[k]) if k in ['width','height','rays_per_pixel','stack','pilot_rays'] else float(row[k])
    for k in ['carrier_mode','edge_score_mode','edge_fusion_mode']:
        if k in row and pd.notna(row[k]):
            overrides[k] = str(row[k])
    for k in ['include_ultrasonic','include_polarization','material_labels']:
        if k in row and pd.notna(row[k]):
            overrides[k] = bool(row[k])
    seed = int(row.get('seed', 42)) if pd.notna(row.get('seed', 42)) else 42
    print('running', run_id, scene, preset, overrides)
    res = llw.run_sensor_preset(prims, preset_name=preset, scene_name=f'{scene}_{run_id:02d}', out_dir=str(run_dir), seed=seed, **overrides)
    diag = dict(res['diagnostics'])
    diag['run_id'] = run_id
    diag['input_scene'] = scene
    diag['contact_sheet'] = res['paths'].get('contact_sheet')
    all_diag.append(diag)
    contact_paths.append(res['paths'].get('contact_sheet'))
    sd = dict(res.get('structure_density') or {})
    sd.update({'run_id': run_id, 'scene': scene, 'preset': preset})
    structure.append(sd)
    for rec in res.get('material_core_agreement') or []:
        rec=dict(rec); rec.update({'run_id': run_id}); material_core.append(rec)
    for rec in res.get('material_filled_agreement') or []:
        rec=dict(rec); rec.update({'run_id': run_id}); material_filled.append(rec)
    for rec in res.get('boundary_adjacency') or []:
        rec=dict(rec); rec.update({'run_id': run_id}); boundary.append(rec)

df = pd.DataFrame(all_diag)
df.to_csv(OUT/'sweep_metrics.csv', index=False)
pd.DataFrame(structure).to_csv(OUT/'structure_density_report.csv', index=False)
pd.DataFrame(material_core).to_csv(OUT/'material_core_agreement.csv', index=False)
pd.DataFrame(material_filled).to_csv(OUT/'material_filled_agreement.csv', index=False)
pd.DataFrame(boundary).to_csv(OUT/'boundary_adjacency_report.csv', index=False)
edge_cols = [c for c in df.columns if c.startswith('edge_') or c in ['run_id','input_scene','preset','geom_edge_frac']]
df[edge_cols].to_csv(OUT/'edge_threshold_diagnostics_summary.csv', index=False)
df[edge_cols].head(10)


In [ ]:
from PIL import Image, ImageDraw
imgs=[]
for p in contact_paths[:6]:
    if p and os.path.exists(p):
        im=Image.open(p).convert('RGB')
        w,h=im.size
        scale=min(520/w, 1.0)
        imgs.append(im.resize((int(w*scale), int(h*scale))))
if imgs:
    cols=2
    W=max(im.width for im in imgs)*cols+30*(cols+1)
    rows=(len(imgs)+cols-1)//cols
    H=sum(max(imgs[i].height for i in range(r*cols, min((r+1)*cols, len(imgs)))) for r in range(rows))+30*(rows+1)
    canvas=Image.new('RGB',(W,H),(8,8,10))
    x0=y=30
    for i,im in enumerate(imgs):
        r=i//cols; c=i%cols
        x=30+c*(max(m.width for m in imgs)+30)
        # compute y by previous row heights
        y=30
        for rr in range(r):
            y += max(imgs[j].height for j in range(rr*cols, min((rr+1)*cols,len(imgs))))+30
        canvas.paste(im,(x,y))
    canvas.save(OUT/'top_contact_sheets.png')
    display(canvas)
else:
    print('No contact sheets found.')


In [ ]:
import zipfile
zip_path = Path('lidar_wave_test_outputs_v0610.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob('*'):
        zf.write(p, arcname=p.relative_to(OUT.parent))
print(zip_path)
files.download(str(zip_path))
